# Gemma 4 Developer Agent — Post-Run Trace Analytics

Run this notebook once to generate two complementary views from the same normalized traces: a single-attempt explorer and a whole-batch report. Analysis is offline and read-only; no code from traces is executed. Open `../reports/itr-1/report.html` for one trace or `../reports/itr-1/batch_report.html` for the batch.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'agent_eval').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'agent_eval').is_dir():
    raise RuntimeError('Run this notebook from gemma4_eval or its notebooks folder')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DEFAULT_TRACE_DIR = next(
    (path for path in [PROJECT_ROOT.parent / 'logs/remote/akshay/itr-1',
                      Path('/Users/akshay/kaggle/wt/main/logs/remote/akshay/itr-1')]
     if path.is_dir()),
    PROJECT_ROOT.parent / 'logs/remote/akshay/itr-1',
)
roots_file = os.environ.get('GEMMA_REPO_ROOTS_JSON')
CONFIG = {
    'trace_dir': os.environ.get('GEMMA_TRACE_DIR', str(DEFAULT_TRACE_DIR)),
    'trace_globs': os.environ.get('GEMMA_TRACE_GLOBS', '**/model_trace.json,**/trace_*.json').split(','),
    'gold_dir': os.environ.get('GEMMA_GOLD_DIR'),
    'repo_roots': json.loads(Path(roots_file).read_text()) if roots_file else {},
    'compare_dir': None,
    'output_dir': os.environ.get('GEMMA_OUTPUT_DIR', str(PROJECT_ROOT / 'reports' / 'itr-1')),
    'experiment_id': os.environ.get('GEMMA_EXPERIMENT_ID', 'itr-1'),
    'strict_schema': False,
    'redact_text': True,
    'include_reasoning_in_viewer': False,
}

## Load and normalize

Malformed inputs are reported per file. Unknown outcomes and unmatched results remain unknown. Duplicates remain separate attempts.

In [ ]:
from agent_eval import load_experiment, compute_metrics, classify_failures, export_report
from agent_eval.plots import render_report

experiment = load_experiment(
    CONFIG['trace_dir'], experiment_id=CONFIG['experiment_id'],
    trace_globs=CONFIG['trace_globs'], strict_schema=CONFIG['strict_schema'],
    redact_text=CONFIG['redact_text'],
    include_reasoning_in_viewer=CONFIG['include_reasoning_in_viewer'],
)
metrics = compute_metrics(experiment, gold_dir=CONFIG['gold_dir'],
                          repo_roots=CONFIG['repo_roots'], compare_dir=CONFIG['compare_dir'])
findings = classify_failures(experiment, metrics)
from agent_eval.review.entry import review_notebook
metrics = review_notebook(experiment, metrics, findings, CONFIG)
metrics.summary

## Single-attempt explorer

Select a task to inspect its timeline, tool calls and results, budget, agent-side tests, independent final verification, token usage, final patch, findings, and source evidence. Batch comparisons appear in the separate section below.

In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
report = render_report(experiment, metrics, findings)

## Export reproducible artifacts

This cell writes both HTML reports and batch summary tables, alongside the normalized CSV and JSONL tables. The HTML files bundle Plotly and work offline.

In [ ]:
output = export_report(experiment, metrics, findings, CONFIG['output_dir'])
print(f'Exports: {output}')
print(f'Single-attempt report: {output / "report.html"}')
print(f'Whole-batch report: {output / "batch_report.html"}')

## Metric availability and optional modules

Gold-function recall needs gold patches and reliable baseline symbol maps. Paired outcomes need a second batch. Missing prerequisites never become zero-valued scores.

In [ ]:
display(metrics.metrics.groupby(['metric_id', 'availability']).size().rename('attempts').reset_index())
display(metrics.modules)

## Whole-batch report

The same notebook now produces a second report focused on aggregate outcomes, source-edit and verification coverage, budget exhaustion, repeated submissions, tool reliability, time/tokens, and data quality. All charts come from the normalized tables above. The separate HTML file is `../reports/itr-1/batch_report.html`. The task index links back to the detailed trace explorer.

In [ ]:
from agent_eval import summarize_batch
from agent_eval.plots import render_batch_report

batch = summarize_batch(experiment, metrics, findings)
batch_view = render_batch_report(batch)
print(f'Whole-batch HTML: {output / "batch_report.html"}')